# 12 · AI-assisted relabelling (all classes)

Makes clean, consistent labels for every image in train, valid and test, starting from the current (human) labels.
The new labels go to `datasets/lds_v2-coco/`. Nothing in `lds_v1` or Roboflow is changed until you run the upload cell at the end.

| Class | How the new labels are made | What you check |
|---|---|---|
| leaf | **SAM 2** redraws each labelled leaf tightly from its box; kept only if it agrees with the old outline. Leaves the trained model sees but nobody labelled are **added** | outlines, added leaves |
| leaf spot | SAM 2 redraws each spot tightly. A **colour check** flags "spots" that aren't darker / browner than the leaf. Spots off every leaf are flagged; confident model spots with no label are **added** | flagged spots, added spots |
| powdery mildew | **Your labels are kept**, cleaned automatically (clipped to leaves, specks removed, spot pixels cut out). Suspicious labels are flagged. (An experimental pixel classifier, `MILDEW_MODE = "classifier"`, confused glossy leaf bumps with powder at full resolution, so it is off.) | whole-leaf labels, labels with nothing whitish or speckled inside, spots that look like dark mildew |

**Test images never use the trained model**, only their human labels (SAM and the pixel classifier, which learns from training images only), so the test labels carry no model bias.

**Labelling rules** (the pipeline follows them; use them when correcting):
- **leaf**: one outline per visible leaf, tight to the edge, including diseased parts.
- **leaf_spot**: brown / tan / grey dead tissue with a solid (not powdery) centre, traced at its outer dark margin; never off a leaf.
- **powdery_mildew**: every patch of fungal growth, at any stage, traced at its outer edge:
  - **early / typical:** white to whitish-grey powdery or felty film, often faint and diffuse, with no sharp edge;
  - **with a ring:** a white or grey centre with a dark ring;
  - **severe / old:** dark, dry, **purplish-grey to purplish-brown** blotches, still with a fine grey speckle of powder or threads, merging into irregular areas.

  Never a whole leaf unless all of it is coated.
- **Telling dark mildew from leaf spot.** Leaf spot (Cercospora) is a small, compact, round spot with a **tan / brown dry centre** and a **sharp reddish-brown or purple border**, smooth inside with no powder. Dark mildew is **greyer** (not reddish), **finely speckled**, **diffuse-edged** and usually larger. Sources:
  - PNW Plant Disease Handbook: "upper surfaces develop yellowish green blotches or turn purplish brown";
  - UConn: Cercospora gives "tan spots with reddish brown borders";
  - in this dataset, spot tissue is much redder than its leaf and smooth; mildew is grey and speckled.

**Order:**
1. Train the mildew classifier.
2. **Calibrate:** look at the pictures and pick `MILDEW_CUTOFF`.
3. Trial on a few images (`LIMIT`).
4. Full run (about 1 hour).
5. Review sheets and CSV.
6. Upload to Roboflow `lds_v2`.
7. Review in Roboflow.

In [ ]:
import collections
import csv
import json
import os
import shutil
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image
from pycocotools.coco import COCO
from sklearn.ensemble import RandomForestClassifier
from ultralytics import SAM, YOLO

import relabel
from eval_utils import CLASSES, paint_class_map

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
SRC_NAME = f"{os.environ['ROBOFLOW_PROJECT']}-v{os.environ['ROBOFLOW_VERSION']}"  # lds_v1-v2
SRC_DIR = ROOT / "datasets" / f"{SRC_NAME}-coco"

## Settings

- `LIMIT`: images per split for a trial (`None` = all). Trial output goes to `datasets/lds_v2-coco-trial/`, so it never mixes with the full run.
- `SETTINGS`: thresholds of the pipeline (see `relabel.Settings`). `mildew_cutoff` is set after the calibration pictures.

In [ ]:
MODEL_CKPT = ROOT / "runs" / "yolo26x-seg_data-v2" / "weights" / "best.pt"  # current best model (proposals, train/valid only)
SAM_WEIGHTS = "sam2.1_l.pt"  # downloaded by Ultralytics on first use
SPLITS = ("train", "valid", "test")
LIMIT = None  # e.g. 8 for a trial run
OVERWRITE = False  # False: a split whose output already exists is skipped (lets a long run resume)

MILDEW_MODE = "keep"  # "keep" (your labels, cleaned + flagged) or "classifier" (experimental, see the table above)
MILDEW_CUTOFF = 0.5  # classifier mode only: set after looking at the calibration pictures
USE_DINO = False  # classifier mode only: add DINOv2 texture features
SETTINGS = relabel.Settings(mildew_mode=MILDEW_MODE, mildew_cutoff=MILDEW_CUTOFF)
RF_IMAGES_MILDEW, RF_IMAGES_OTHER = 120, 40  # training images for the mildew classifier
SEED = 0

DST_NAME = "lds_v2"
DST_DIR = ROOT / "datasets" / (f"{DST_NAME}-coco" if LIMIT is None else f"{DST_NAME}-coco-trial")
REVIEW = ROOT / "results" / ("relabel" if LIMIT is None else "relabel-trial")
REVIEW.mkdir(parents=True, exist_ok=True)
print("source:", SRC_DIR.relative_to(ROOT), "| output:", DST_DIR.relative_to(ROOT), "| review:", REVIEW.relative_to(ROOT))

SURFACE, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"
MAP_COLORS = ListedColormap(["#f0efec", "#b7d3f6", "#eb6834", "#1baf7a"])  # background, leaf, leaf spot, mildew
plt.rcParams.update({"figure.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.titlecolor": INK,
                     "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"], "font.size": 9,
                     "savefig.bbox": "tight"})

## Data and models

In [ ]:
cocos = {split: COCO(str(SRC_DIR / split / "_annotations.coco.json")) for split in SPLITS}
CAT_IDS = {c["name"]: c["id"] for c in cocos["train"].loadCats(cocos["train"].getCatIds())}
for split, coco in cocos.items():
    names = {cid: n for n, cid in CAT_IDS.items()}
    counts = collections.Counter(names.get(a["category_id"]) for a in coco.loadAnns(coco.getAnnIds()))
    print(f"{split:<6} {len(coco.getImgIds()):>5} images | " + ", ".join(f"{n}: {counts[n]}" for n in ("leaf", "leaf_spot", "powdery_mildew")))


def load_image(split, img):
    path = SRC_DIR / split / img["file_name"]
    rgb = np.array(Image.open(path).convert("RGB"))
    coco = cocos[split]
    old = {"leaf": [], "leaf_spot": [], "powdery_mildew": []}
    for a in coco.loadAnns(coco.getAnnIds(imgIds=img["id"])):
        name = coco.cats[a["category_id"]]["name"]
        if name in old:
            old[name].append(coco.annToMask(a).astype(bool))
    return path, rgb, old


model = YOLO(MODEL_CKPT)
sam = SAM(SAM_WEIGHTS)
print("model:", MODEL_CKPT.relative_to(ROOT), "| SAM:", SAM_WEIGHTS)

## 1. Train the mildew pixel classifier (classifier mode only)

Learns what "powder on this leaf" looks like from the training images, with no hand painting:
- **positives**: pixels inside existing mildew labels that are both brighter and greyer than the leaf. Powder whitens and desaturates; pale veins and green inside loose outlines don't do both.
- **negatives**: leaf pixels far (> 40 px) from any mildew label and not on a spot (healthy green, veins, glare, rims).

Features compare each pixel with its own leaf's **healthy** colour (the median of the leaf's darker half), so lighting, leaf shade and heavily coated leaves are handled. They also include distance to the leaf edge, and optionally DINOv2 texture features (`USE_DINO`; see `relabel.feature_names`).
20 % of the images are held out to check the classifier on pixels it never saw.

In [ ]:
clf = dino = None
if MILDEW_MODE != "classifier":
    print("MILDEW_MODE = 'keep': no classifier needed")
else:
    rng = np.random.default_rng(SEED)
    train_imgs = cocos["train"].loadImgs(cocos["train"].getImgIds())
    has_mildew = {a["image_id"] for a in cocos["train"].loadAnns(cocos["train"].getAnnIds(catIds=[CAT_IDS["powdery_mildew"]]))}
    mildew_imgs = [im for im in train_imgs if im["id"] in has_mildew]
    other_imgs = [im for im in train_imgs if im["id"] not in has_mildew]
    chosen = [mildew_imgs[i] for i in rng.choice(len(mildew_imgs), min(RF_IMAGES_MILDEW, len(mildew_imgs)), replace=False)] + \
             [other_imgs[i] for i in rng.choice(len(other_imgs), min(RF_IMAGES_OTHER, len(other_imgs)), replace=False)]
    rng.shuffle(chosen)
    dino = relabel.DinoFeatures().fit([load_image("train", im)[1] for im in chosen[:40]]) if USE_DINO else None

    Xs, ys, groups = [], [], []
    t0 = time.time()
    for k, img in enumerate(chosen):
        _, rgb, old = load_image("train", img)
        leaves = [relabel.fill_holes(m) for m in old["leaf"] if m.any()]
        mildew = np.logical_or.reduce(old["powdery_mildew"]) if old["powdery_mildew"] else np.zeros(rgb.shape[:2], bool)
        spots = np.logical_or.reduce(old["leaf_spot"]) if old["leaf_spot"] else np.zeros(rgb.shape[:2], bool)
        X, y = relabel.training_pixels(rgb, leaves, mildew, spots, rng, dino=dino)
        if X is not None:
            Xs.append(X); ys.append(y); groups.append(np.full(len(y), k))
    X, y, groups = np.concatenate(Xs), np.concatenate(ys), np.concatenate(groups)
    print(f"{len(chosen)} images → {len(y)} pixels ({int(y.sum())} mildew) in {time.time() - t0:.0f} s")

    held_out = groups >= int(0.8 * len(chosen))
    clf = RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5, class_weight="balanced_subsample",
                                 n_jobs=-1, random_state=SEED)
    clf.fit(X[~held_out], y[~held_out])
    p = clf.predict(X[held_out])
    tp, fp, fn = ((p == 1) & (y[held_out] == 1)).sum(), ((p == 1) & (y[held_out] == 0)).sum(), ((p == 0) & (y[held_out] == 1)).sum()
    print(f"held-out images: mildew precision {tp / max(tp + fp, 1):.3f}, recall {tp / max(tp + fn, 1):.3f} "
          f"(clear cases only; the real test is the pictures below)")
    clf.fit(X, y)  # final classifier on all chosen images
    joblib.dump(clf, REVIEW / "mildew_classifier.joblib")
    print("feature importance:", ", ".join(f"{f} {v:.2f}" for f, v in sorted(zip(relabel.feature_names(dino), clf.feature_importances_), key=lambda t: -t[1])))

## 2. Calibrate the mildew cut-off (classifier mode only; look at these pictures)

The validation images with the most mildew, at three cut-offs. Pick the one where the green covers the visible white powder (including faint films) and nothing on healthy green, then set `MILDEW_CUTOFF` in the settings cell and re-run from there.
- Probability map: dark = surely not mildew, bright = surely mildew; only computed inside the search region.

In [ ]:
if MILDEW_MODE != "classifier":
    print("MILDEW_MODE = 'keep': no calibration needed")
else:
    CUTOFFS = (0.3, 0.5, 0.7)
    valid_imgs = cocos["valid"].loadImgs(cocos["valid"].getImgIds())
    mildew_px = {}
    for a in cocos["valid"].loadAnns(cocos["valid"].getAnnIds(catIds=[CAT_IDS["powdery_mildew"]])):
        mildew_px[a["image_id"]] = mildew_px.get(a["image_id"], 0) + a["area"]
    calib = sorted(valid_imgs, key=lambda im: -mildew_px.get(im["id"], 0))[:8]

    fig, axes = plt.subplots(len(calib), 3 + len(CUTOFFS), figsize=(3.3 * (3 + len(CUTOFFS)), 3.3 * len(calib)), squeeze=False)
    for axr, img in zip(axes, calib):
        path, rgb, old = load_image("valid", img)
        res, _, _ = relabel.relabel_image(path, rgb, old, sam, clf, model, SETTINGS, keep_prob=True, dino=dino)
        old_mildew = np.logical_or.reduce(old["powdery_mildew"])
        axr[0].imshow(rgb); axr[0].set_title(img["file_name"][:26], fontsize=8)
        over = rgb.copy(); over[old_mildew] = (0.5 * over[old_mildew] + 0.5 * np.array([27, 175, 122])).astype(np.uint8)
        axr[1].imshow(over); axr[1].set_title("old label", fontsize=8)
        axr[2].imshow(np.nan_to_num(res.mildew_prob, nan=0), cmap="gray", vmin=0, vmax=1); axr[2].set_title("probability", fontsize=8)
        leaf_union = np.logical_or.reduce(res.leaves) if res.leaves else np.zeros(rgb.shape[:2], bool)
        spot_px = np.logical_or.reduce(res.spots) if res.spots else np.zeros(rgb.shape[:2], bool)
        for ax, c in zip(axr[3:], CUTOFFS):
            m = relabel.clean_disease(np.nan_to_num(res.mildew_prob, nan=0) >= c, SETTINGS.min_disease_px, SETTINGS.max_hole_px,
                                      smooth=True) & leaf_union & ~spot_px
            over = rgb.copy(); over[m] = (0.45 * over[m] + 0.55 * np.array([27, 175, 122])).astype(np.uint8)
            ax.imshow(over); ax.set_title(f"cut-off {c}", fontsize=8)
        for ax in axr:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(REVIEW / "calibration_mildew.png", dpi=90)
    plt.show()

## 3. Relabel every image

Writes `DST_DIR/<split>/` (image hard links, or copies if links aren't possible, plus `_annotations.coco.json`) and collects a review row per image.
Takes about 1 s per image (SAM + classifier). Test images use no model (`model=None`).

In [ ]:
def link(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


rows = []
for split in SPLITS:
    out_dir = DST_DIR / split
    out_json = out_dir / "_annotations.coco.json"
    if out_json.exists() and not OVERWRITE:
        print(f"{split}: exists, skipped (OVERWRITE = False)")
        rows += json.loads((REVIEW / f"review_{split}.json").read_text()) if (REVIEW / f"review_{split}.json").exists() else []
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    coco = cocos[split]
    imgs = coco.loadImgs(coco.getImgIds())[: LIMIT or None]
    src = json.loads((SRC_DIR / split / "_annotations.coco.json").read_text())
    new_anns, next_id, split_rows = [], 1, []
    t0 = time.time()
    for k, img in enumerate(imgs):
        path, rgb, old = load_image(split, img)
        res, old_cov, new_cov = relabel.relabel_image(path, rgb, old, sam, clf, None if split == "test" else model, SETTINGS,
                                                      dino=dino)
        anns, next_id = relabel.to_annotations(res, img["id"], CAT_IDS, next_id)
        new_anns += anns
        link(path, out_dir / img["file_name"])
        change = sum(abs((new_cov or {}).get(d, 0) - (old_cov or {}).get(d, 0)) for d in ("leaf_spot", "powdery_mildew"))
        split_rows.append({"split": split, "file": img["file_name"], "image_id": img["id"],
                           "old_spot_pct": round((old_cov or {}).get("leaf_spot", 0), 2),
                           "new_spot_pct": round((new_cov or {}).get("leaf_spot", 0), 2),
                           "old_mildew_pct": round((old_cov or {}).get("powdery_mildew", 0), 2),
                           "new_mildew_pct": round((new_cov or {}).get("powdery_mildew", 0), 2),
                           "old_leaves": len(old["leaf"]), "new_leaves": len(res.leaves),
                           "old_spots": len(old["leaf_spot"]), "new_spots": len(res.spots),
                           "flags": ";".join(f"{k_}={v}" for k_, v in sorted(res.flags.items())),
                           "score": round(change + sum(res.flags.values()) * 0.5, 2)})
        if (k + 1) % 100 == 0 or k + 1 == len(imgs):
            print(f"{split}: {k + 1}/{len(imgs)} images, {(time.time() - t0) / (k + 1):.2f} s per image", flush=True)
    src["images"] = imgs
    src["annotations"] = new_anns
    out_json.write_text(json.dumps(src))
    (REVIEW / f"review_{split}.json").write_text(json.dumps(split_rows))
    rows += split_rows
print("done:", len(rows), "images")

## 4. Review list and sheets

- `review.csv`: one row per image, sorted by **score** (coverage change + number of flags), so the images most worth checking come first.
- **Flags:**
  - `leaf_speck_removed`: tiny "leaf" labels (under 100 px) were dropped.
  - `leaf_sam_rejected`: SAM disagreed with a leaf outline; the old outline was kept, so check that leaf.
  - `leaf_added`: a leaf with no label was added.
  - `spot_sam_rejected`: same as `leaf_sam_rejected`, for a spot.
  - `spot_colour`: a "spot" that isn't darker or browner than the leaf.
  - `spot_off_leaf`: a spot not on any leaf.
  - `spot_added`: a model spot with no label was added.
  - `mildew_big_change` / `spot_big_change`: coverage changed by more than 2 pp.
  - `mildew_label_not_white`: an old mildew label with almost no white pixels inside, so possibly not mildew.
  - `mildew_whole_leaf`: an old mildew label covering most of a leaf.
- Contact sheets of the top images: photo | old labels | new labels | changes (yellow = disease added, red = disease removed, blue = leaf outline changed).

In [ ]:
rows.sort(key=lambda r: -r["score"])
with open(REVIEW / "review.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)
flag_counts = collections.Counter()
for r in rows:
    for item in filter(None, r["flags"].split(";")):
        name, n = item.split("=")
        flag_counts[name] += int(n)
print("review.csv:", len(rows), "images |", sum(1 for r in rows if r["flags"]), "flagged")
for name, n in flag_counts.most_common():
    print(f"  {name:<24} {n}")

new_cocos = {split: COCO(str(DST_DIR / split / "_annotations.coco.json")) for split in SPLITS}


def class_map(coco, img):
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
    return paint_class_map([coco.annToMask(a).astype(bool) for a in anns], [coco.cats[a["category_id"]]["name"] for a in anns],
                           (img["height"], img["width"]))


def change_map(old_map, new_map):
    rgb = np.full(old_map.shape + (3,), 0xf0, np.uint8)
    rgb[(old_map > 0) | (new_map > 0)] = (0xe1, 0xe0, 0xd9)
    rgb[(old_map > 0) != (new_map > 0)] = (0x2a, 0x78, 0xd6)  # leaf outline changed
    rgb[(new_map >= 2) & (new_map != old_map)] = (0xfa, 0xb2, 0x19)  # disease added / class changed
    rgb[(old_map >= 2) & (new_map < 2)] = (0xd0, 0x3b, 0x3b)  # disease removed
    return rgb


TOP, PER_SHEET = 48, 6
for s_i in range(0, min(TOP, len(rows)), PER_SHEET):
    chunk = rows[s_i:s_i + PER_SHEET]
    fig, axes = plt.subplots(len(chunk), 4, figsize=(15, 3.8 * len(chunk)), squeeze=False)
    for axr, r in zip(axes, chunk):
        img = cocos[r["split"]].loadImgs(r["image_id"])[0]
        old_map, new_map = class_map(cocos[r["split"]], img), class_map(new_cocos[r["split"]], img)
        axr[0].imshow(Image.open(SRC_DIR / r["split"] / r["file"]).convert("RGB"))
        axr[0].set_title(f"{r['split']}: {r['file'][:28]}\n{r['flags'] or 'no flags'}", fontsize=7)
        axr[1].imshow(old_map, cmap=MAP_COLORS, vmin=0, vmax=3, interpolation="nearest")
        axr[1].set_title(f"old: spot {r['old_spot_pct']} % · mildew {r['old_mildew_pct']} %", fontsize=8)
        axr[2].imshow(new_map, cmap=MAP_COLORS, vmin=0, vmax=3, interpolation="nearest")
        axr[2].set_title(f"new: spot {r['new_spot_pct']} % · mildew {r['new_mildew_pct']} %", fontsize=8)
        axr[3].imshow(change_map(old_map, new_map), interpolation="nearest"); axr[3].set_title("changes", fontsize=8)
        for ax in axr:
            ax.axis("off")
    fig.legend(handles=[Patch(color="#b7d3f6", label="leaf"), Patch(color="#eb6834", label="leaf spot"),
                        Patch(color="#1baf7a", label="powdery mildew"), Patch(color="#fab219", label="disease added"),
                        Patch(color="#d03b3b", label="disease removed"), Patch(color="#2a78d6", label="leaf outline changed")],
               loc="upper center", ncols=6, bbox_to_anchor=(0.5, 1.01))
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    fig.savefig(REVIEW / f"sheet_{s_i // PER_SHEET + 1:02d}.png", dpi=80)
    if s_i == 0:
        plt.show()
    else:
        plt.close(fig)
print("sheets saved in", REVIEW.relative_to(ROOT))

## 5. Checks

In [ ]:
for split in SPLITS:
    old_c, new_c = cocos[split], new_cocos[split]
    n_old = len(old_c.getImgIds()[: LIMIT or None])
    assert len(new_c.getImgIds()) == n_old, f"{split}: image count changed"
    assert all((DST_DIR / split / im["file_name"]).exists() for im in new_c.loadImgs(new_c.getImgIds())), "missing image file"
    ids = set(new_c.getImgIds())
    count = lambda c, name: sum(1 for a in c.loadAnns(c.getAnnIds(catIds=[CAT_IDS[name]])) if a["image_id"] in ids)
    print(f"{split:<6} {n_old} images | " + " | ".join(f"{n}: {count(old_c, n)} → {count(new_c, n)}"
                                                       for n in ("leaf", "leaf_spot", "powdery_mildew")))
    bad = [a for a in new_c.loadAnns(new_c.getAnnIds()) if len(a["segmentation"][0]) < 6]
    assert not bad, f"{split}: {len(bad)} polygons with < 3 points"
print("checks passed")

## 6. Label agreement (label quality for the paper)

Relabel the same 20 images twice independently. Either a second person draws the corrections, or you redo them a week later. Then export both passes from Roboflow as COCO and set the two paths here. The output is the per-class pixel IoU between the passes. Targets: leaf ≥ 0.95, leaf spot ≥ 0.85, mildew ≥ 0.85.

In [ ]:
PASS_A = None  # e.g. ROOT / "datasets" / "agreement_pass_a" / "_annotations.coco.json"
PASS_B = None


def agreement(path_a, path_b):
    a, b = COCO(str(path_a)), COCO(str(path_b))
    by_name = {im["file_name"]: im for im in b.loadImgs(b.getImgIds())}
    inter = collections.Counter(); union = collections.Counter()
    for img in a.loadImgs(a.getImgIds()):
        if img["file_name"] not in by_name:
            continue
        ma, mb = class_map(a, img), class_map(b, by_name[img["file_name"]])
        for i, c in enumerate(CLASSES[1:], 1):
            inter[c] += int(((ma == i) & (mb == i)).sum()); union[c] += int(((ma == i) | (mb == i)).sum())
    return {c: inter[c] / union[c] for c in union if union[c]}


if PASS_A and PASS_B:
    for c, v in agreement(PASS_A, PASS_B).items():
        print(f"{c:<16} agreement IoU {v:.3f}")
else:
    print("set PASS_A and PASS_B after the double labelling")

## 7. Upload to Roboflow (after reviewing the sheets)

- `UPLOAD = True`: train + valid + test go to a **new project `lds_v2`** (instance segmentation), keeping the splits. `lds_v1` stays as it is.
- `UPLOAD_TEST_TO_LDS_TEST = True` additionally **replaces the labels** of the matching images in `lds_test`. **Careful: this overwrites any corrections you've already made there.** Leave it off if you're editing `lds_test` by hand.

In Roboflow, review in this order:
1. The flagged images (top of `review.csv`).
2. All other images, at a quicker pace.

Then generate `lds_v2` version 1: Auto-Orient, Resize 1024 (Stretch), **no augmentation**, keep the splits.

In [ ]:
UPLOAD = False
UPLOAD_TEST_TO_LDS_TEST = False

if UPLOAD or UPLOAD_TEST_TO_LDS_TEST:
    assert LIMIT is None, "upload only the full run, not a trial"
    from roboflow import Roboflow

    ws = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"]).workspace(os.environ["ROBOFLOW_WORKSPACE"])
    if UPLOAD:
        for split in SPLITS:
            ws.upload_dataset(str(DST_DIR / split), DST_NAME, project_type="instance-segmentation",
                              batch_name=f"relabel-{split}", num_retries=2, split=split)
    if UPLOAD_TEST_TO_LDS_TEST:
        ws.upload_dataset(str(DST_DIR / "test"), "lds_test", project_type="instance-segmentation",
                          batch_name="relabel-test", num_retries=2, split="test", annotation_overwrite=True)
    for p in Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"]).workspace(os.environ["ROBOFLOW_WORKSPACE"]).project_list:
        if p["id"].split("/")[-1] in (DST_NAME, "lds_test"):
            print(p["id"], p.get("images"), "images |", p.get("classes"))
else:
    print("UPLOAD is off")